To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://github.com/unslothai/unsloth#installation-instructions---conda).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save) (eg for Llama.cpp).

* We support Llama, Mistral, CodeLlama, TinyLlama, Vicuna, Open Hermes etc
* And Yi, Qwen ([llamafied](https://huggingface.co/models?sort=trending&search=qwen+llama)), Deepseek, all Llama, Mistral derived archs.
* We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
* `max_seq_length` can be set to anything, since we do automatic RoPE Scaling via [kaiokendev's](https://kaiokendev.github.io/til) method.
* With [PR 26037](https://github.com/huggingface/transformers/pull/26037), we support downloading 4bit models **4x faster**! [Our repo](https://huggingface.co/unsloth) has Llama, Mistral 4bit models.
* [**NEW**] We make Gemma 6 trillion tokens **2.5x faster**! See our [Gemma notebook](https://colab.research.google.com/drive/10NbwlsRChbma1v55m8LAPYG15uQv6HLo?usp=sharing)
* [**NEW**] We make Llama-3 15 trillion tokens **2x faster**! See our [Llama-3 notebook](https://colab.research.google.com/drive/135ced7oHytdxu3N2DNe1Z0kqjyYIkDXp?usp=sharing)

In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/codellama-7b-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.572 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 256, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.10.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `ChatML` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [3]:
from datasets import load_dataset
# dataset = load_dataset("bnadimi/PyraNet-Verilog", split = "train")
dataset = load_dataset('json', data_files='pyranet-no-error.json', split = "train")

In [4]:
# dataset = dataset.select(range(80,90))

In [5]:
len(dataset)

357715

In [24]:
dataset.to_json('pyranet-no-error-select.json')

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

627

In [26]:
print(dataset['code'][0])

module half_adder (
  input A,
  input B,
  output S,
  output C
);

assign S = A ^ B; // bitwise XOR operation to get sum
assign C = A & B; // bitwise AND operation to get carry

endmodule


In [27]:
print(dataset['description'][0])

{"description": "The Verilog code implements a half-adder, which takes two single-bit inputs (A and B) and produces two outputs: S (the sum) and C (the carry). The sum is calculated using the bitwise XOR operation, and the carry is calculated using the bitwise AND operation.", "rank": "20", "complexity": "Intermediate", "compile_status": "No error!", "compile_results": ""}


In [2]:
dataset_reduce = dataset.select(range(50))

In [3]:
len(dataset_reduce)

50

In [4]:
dataset_reduce.to_json('pyranet-no-error-10.json')

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

22343

#### Dataset Explore

In [5]:
dataset_description

NameError: name 'dataset_description' is not defined

In [ ]:
a = '{"description": "The Verilog code defines a module named \\"MyNot\\" that implements a NOT gate. It takes a single input `a` and produces an output `z`, which is the logical negation of `a` (`z = NOT a`).", "rank": "20", "complexity": "Basic", "compile_status": "No error!", "compile_results": ""}'
# a
json.loads(a)

In [ ]:
import json
compile_status_set = set()
compile_status_no_error_count = 0
for i in range(len(dataset)):# :# range(5)
    dataset_description = dataset['description'][i]
    dataset_description = dataset_description.replace("\\\\", "\\")
    
    try:
        des_dict = json.loads(dataset_description)
    except:
        print(f"Error description[{i}]: ", dataset_description)
        input("Dictionary parsing error")
        continue
    cur_compile_status = des_dict['compile_status']
    if cur_compile_status not in compile_status_set:
        print(f"New compile_status in item[{i}]: ", cur_compile_status)
        compile_status_set.add(cur_compile_status)
    if cur_compile_status == 'No error!':
        compile_status_no_error_count += 1
compile_status_set

In [ ]:
compile_status_no_error_count

In [ ]:
filter_dataset = dataset.filter(lambda example: '"compile_status": "No error!"' in example['description'])

In [ ]:
len(filter_dataset)

In [ ]:
filter_dataset.to_json('pyranet-no-error')

#### Dataset Construction

In [ ]:
import json, re
alpaca_prompt = """Below is an instruction that describes a Verilog module, paired with a module definition that provides further context for input and output ports. Write a response that only completes the code of Verilog module, without other descriptions.

### Instruction:
{}

### Module Definition:
{}

### Response:
{}"""
dataset_iter = dataset.iter(batch_size=1)
for example in dataset_iter:
    example_descriptions = example["description"]
    example_codes        = example["code"]
    texts = []
    for example_description, example_code in zip(example_descriptions, example_codes):
        example_description = example_description.replace("\\\\", "\\")
        example_description_dict = json.loads(example_description)

        line_dect = ""
        line_start = -1
        for line in example_code.splitlines():
            module_substr_start = line.find("module")
            if module_substr_start == 0:
                line_dect = line
                break
            elif module_substr_start > 0:
                print("Here: ", line, bytearray(line[:module_substr_start].encode("utf-8")), )
                line_dect = line
            # else:
            #     print("Unknow module start: ", module_substr_start)
                
        semicon_substr_start = example_code.find(";") # module a();
        if (not (module_substr_start >= 0)) or (not (semicon_substr_start > 10)):
            raise Exception(f"Not good module def: {example_code}, {line_dect[:module_substr_start]}")
        # else:
        #     print("Ok: ", example_code[:semicon_substr_start + 1])
        #     print("Ok: ", example_code[semicon_substr_start + 1:])
    # break

In [18]:
import json, re, tempfile, subprocess
import numpy as np
alpaca_prompt = """Below is an instruction that describes a Verilog module, paired with a module definition that provides further context for input and output ports. Write a response that only completes the code of Verilog module, without other descriptions.

### Instruction:
{}

### Module Definition:
{}

### Response:
{}"""
dataset_iter = dataset.iter(batch_size=1)
for example in [{'code': ['`timescale 1ns/1ps module full(a,b,cin,s,cout); input a,b,cin;\noutput s,cout;\n\nassign s = a^b^cin;\nassign cout = (a & b ) | (b & cin ) | (cin & a ) ; endmodule\n'], 'description': ['{"description": "The Verilog code implements a full adder circuit, which takes three inputs (two single-bit values `a` and `b`, and a carry-in `cin`) and produces two outputs: the sum `s` (calculated as the XOR of the three inputs) and the carry-out `cout` (calculated based on the logic of the input bits).", "rank": "17", "complexity": "Intermediate", "compile_status": "No error!", "compile_results": ""}']}]:
# for example in dataset_iter:
    example_descriptions = example["description"]
    example_codes        = example["code"]
    texts = []
    for example_description, example_code in zip(example_descriptions, example_codes):
        example_code = example_code.replace("\\\\", "\\")
        # format, for \n placements
        with tempfile.NamedTemporaryFile(delete=False, suffix=".v") as fp:
            fp.write(example_code.encode(encoding='utf-8'))
            fp.close()
            sub_run = subprocess.run(['verible-verilog-format', fp.name], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
            # example_code = sub_run.stdout.decode("utf-8")
            with open(fp.name, 'r') as file:
                example_code = file.read()
        
        example_description = example_description.replace("\\\\", "\\")
        example_description_dict = json.loads(example_description)

        regex_stage = 0
        regexes = [r'^\s*\(\*.*\*\)', r'^\s*module\s+', r'^\s*((//)|(/\*))']
        code_lines = example_code.splitlines()
        module_def_span = np.array([None, None, None, None])
        num_indexs = 0
        for i in range(len(code_lines)):
            line = code_lines[i] + "\n"
            if regex_stage == 0: # (* attribute *), only one!
                while True:
                    attribute_match = re.match(regexes[regex_stage], line)
                    if attribute_match != None:
                        print("Attribute found: ", line, fp.name)
                        line = line.replace(attribute_match[0], "")
                        num_indexs += len(attribute_match[0])
                        print("Attribute replace: ", line)
                    else:
                        break
                regex_stage += 1
            if regex_stage == 1:
                module_match = re.match(regexes[regex_stage], line)
                if module_match != None:
                    regex_stage += 1
                    span = module_match.span()
                    module_def_span = np.vstack((module_def_span, [span[0] + num_indexs, span[1] + num_indexs, None, None]))
                else:
                    regex_stage -= 1
            if regex_stage == 2:
                if re.match(regexes[regex_stage], line) == None: # Not comment line
                    semicon_pos = line.find(";")
                    if semicon_pos != -1:
                        module_def_span[-1][-2] = num_indexs + semicon_pos + 1
                        # break
                        regex_stage += 1
            if regex_stage == 3:
                if re.match(regexes[regex_stage - 1], line) == None: # Not comment line
                        endmodule_pos = line.find("endmodule")
                        if endmodule_pos != -1:
                            module_def_span[-1][-1] = num_indexs + endmodule_pos + 1
                            regex_stage = 0
                            
            num_indexs += len(line) + (line[-1] != "\n")

        # if module_def_span == 0:
        #     raise Excaption(f"Error: {example_code}")
        if len(module_def_span.shape) < 2:
            raise Exception(f"Module span is all null {example},  {fp.name}, {len(example_code)}: {example_code}")
        module_def_span = np.delete(module_def_span, 0, 0)
        module_wrapper = ""
        for span in module_def_span:
            if None in span:
                raise Exception("Span Error!!")
            module_wrapper += f"""<hdl>
<module_definition>
{example_code[span[0]:span[2]]}
</module_definition>
<module_code>
{example_code[span[2]:span[3]+8]}

</module_code>
</hdl>"""
        module_wrapper = "<hdls>\n" + module_wrapper + "\n</hdls>"
        subprocess.run(['rm', '-f', fp.name])
    # if span == 
    # if len(module_def_span) > 1:
    #     print(module_wrapper)
    #     break
    # break

Attribute found:  (* blackbox *)
 /tmp/tmpwo4gvx4a.v
Attribute replace:  

Attribute found:  (* blackbox *)
 /tmp/tmpwo4gvx4a.v
Attribute replace:  

Attribute found:  (* blackbox *)
 /tmp/tmpwo4gvx4a.v
Attribute replace:  

Attribute found:  (* blackbox *)
 /tmp/tmpwo4gvx4a.v
Attribute replace:  

Attribute found:  (* abc9_lut=2, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* abc9_lut=1, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* abc9_lut=1, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* abc9_lut=1, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* abc9_lut=1, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* abc9_lut=1, lib_whitebox *)
 /tmp/tmpz44spvrh.v
Attribute replace:  

Attribute found:  (* blackbox *)
 /tmp/tmp6bij1hug.v
Attribute replace:  

Attribute found:  (* blackbox *)
 /tmp/tmp6bij1hug.v
Attribute replace:  

Attr

Exception: Module span is all null {'code': ['`timescale 1ns/1ps module full(a,b,cin,s,cout); input a,b,cin;\noutput s,cout;\n\nassign s = a^b^cin;\nassign cout = (a & b ) | (b & cin ) | (cin & a ) ; endmodule\n'], 'description': ['{"description": "The Verilog code implements a full adder circuit, which takes three inputs (two single-bit values `a` and `b`, and a carry-in `cin`) and produces two outputs: the sum `s` (calculated as the XOR of the three inputs) and the carry-out `cout` (calculated based on the logic of the input bits).", "rank": "17", "complexity": "Intermediate", "compile_status": "No error!", "compile_results": ""}']},  /tmp/tmplhk0a7kb.v, 160: `timescale 1ns/1ps module full(a,b,cin,s,cout); input a,b,cin;
output s,cout;

assign s = a^b^cin;
assign cout = (a & b ) | (b & cin ) | (cin & a ) ; endmodule


In [17]:
module_wrapper

'<hdls>\n<hdl>\n<module_definition>\n module BIT_SIFT(add_in, add_out);\n</module_definition>\n<module_code>\n\n    input [31:0] add_in;\n    output [31:0] add_out;\n    \n    assign add_out = add_in << 2;\n    \nendmodule\n\n</module_code>\n</hdl>\n</hdls>'

In [ ]:
import os

# Get the number of logical CPU cores in the system
num_cores = os.cpu_count()
num_cores

In [30]:
import json, re, tempfile, subprocess
# from tqdm import tqdm
alpaca_prompt = """Below is an instruction that describes a Verilog module, paired with a module definition that provides further context for input and output ports. Write a response that only completes the code of Verilog module, without other descriptions.

### Instruction:
{}

### Module Definition:
{}

### Response:
{}"""


EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples, rank):
    example_descriptions = examples["description"]
    example_codes        = examples["code"]
    texts = []
    
    for example_description, example_code in zip(example_descriptions, example_codes):
        # format, for \n placements
        with tempfile.NamedTemporaryFile(delete=False, suffix=".v") as fp:
            fp.write(example_code.encode(encoding='utf-8'))
            fp.close()
            sub_run = subprocess.run(['verible-verilog-format', fp.name], stdout=subprocess.PIPE)
            example_code = sub_run.stdout.decode("utf-8")
        
        example_description = example_description.replace("\\\\", "\\")
        example_description_dict = json.loads(example_description)

        regex_stage = 0
        regexes = [r'^\s*module\s+\w+', r'^\s*((//)|(/\*))']
        code_lines = example_code.splitlines()
        module_def_span = []
        num_indexs = 0
        for i in range(len(code_lines)):
            line = code_lines[i]
            if regex_stage == 0:
                module_match = re.match(regexes[0], line)
                if module_match != None:
                    regex_stage = 1
                    span = module_match.span()
                    module_def_span.append([span[0] + num_indexs, span[1] + num_indexs, None])
            if regex_stage == 1:
                if re.match(regexes[1], line) == None: # Not comment line
                    semicon_pos = line.find(";")
                    if semicon_pos != -1:
                        module_def_span[-1][-1] = num_indexs + semicon_pos + 1
                        break
            num_indexs += len(line) + 1
        
        if module_semicon_index == None or module_def_span == None:
            raise Exception(f"Fail in module, rank {rank}, description {example_description_dict['description']} \n###### Code : {example_code}")
        module_def_ex = example_code[module_def_span[0]:module_semicon_index]
        #
        instruction  = example_description_dict['description'] + "\nPlace the completion of the Verilog module in an XML tag <code></code>"
        module_def   = module_def_ex
        output       = f'''<code>
{example_code[module_semicon_index+1:]}
</code>'''
    
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, module_def, output) + EOS_TOKEN
        texts.append(text)
        
    return { "text" : texts, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True, num_proc=18, with_rank=True)

Map (num_proc=18):   0%|          | 0/357715 [00:00<?, ? examples/s]

TypeError: slice indices must be integers or None or have an __index__ method

In [ ]:
dataset['text'][0]

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Set num_train_epochs = 1 for full training runs
        num_train_epochs = 1,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    ),
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q5_k_m", token = "")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/u54VK8m8tk) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Zephyr DPO 2x faster [free Colab](https://colab.research.google.com/drive/15vttTpzzVXv_tJwEk-hIcQ0S9FcEWvwP?usp=sharing)
2. Llama 7b 2x faster [free Colab](https://colab.research.google.com/drive/1lBzz5KeZJKXjvivbYvmGarix9Ao6Wxe5?usp=sharing)
3. TinyLlama 4x faster full Alpaca 52K in 1 hour [free Colab](https://colab.research.google.com/drive/1AZghoNBQaMDgWJpi4RbffGM1h6raLUj9?usp=sharing)
4. CodeLlama 34b 2x faster [A100 on Colab](https://colab.research.google.com/drive/1y7A0AxE3y8gdj4AVkl2aZX47Xu3P1wJT?usp=sharing)
5. Mistral 7b [free Kaggle version](https://www.kaggle.com/code/danielhanchen/kaggle-mistral-7b-unsloth-notebook)
6. We also did a [blog](https://huggingface.co/blog/unsloth-trl) with 🤗 HuggingFace, and we're in the TRL [docs](https://huggingface.co/docs/trl/main/en/sft_trainer#accelerate-fine-tuning-2x-using-unsloth)!
7. `ChatML` for ShareGPT datasets, [conversational notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing)
8. Text completions like novel writing [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing)
9. Gemma 6 trillion tokens is 2.5x faster! [free Colab](https://colab.research.google.com/drive/10NbwlsRChbma1v55m8LAPYG15uQv6HLo?usp=sharing)

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Support our work if you can! Thanks!
</div>